In [3]:
import numpy as np
import pyomo.environ as pyo

# --- Configuración de la Red de 3 Nodos ---
N = 3
Sbase = 1.0 # Los datos ya están en pu

# Impedancia de línea z = 0.01 + j0.1 -> Admitancia y = 1/z
z_line = complex(0.01, 0.1)
y_line = 1 / z_line
g, b = np.real(y_line), np.imag(y_line)

# Construcción de Matriz Ybus (Triangular: 1-2, 1-3, 2-3)
G_mat = np.zeros((N, N))
B_mat = np.zeros((N, N))

lines = [(0, 1), (0, 2), (1, 2)] # Nodos 0, 1, 2 en Python (1, 2, 3 en el problema)
for i, j in lines:
    G_mat[i, i] += g; G_mat[j, j] += g; G_mat[i, j] = -g; G_mat[j, i] = -g
    B_mat[i, i] += b; B_mat[j, j] += b; B_mat[i, j] = -b; B_mat[j, i] = -b

modelo = pyo.ConcreteModel(name='SCOPF_3_Nodos')

# --- Conjuntos y Índices ---
modelo.buses = pyo.Set(initialize=range(N))
modelo.scenarios = pyo.Set(initialize=['base', 'cont']) # Caso base y Contingencia

# --- Variables de Decisión ---
# Variables en cada escenario s
modelo.V = pyo.Var(modelo.scenarios, modelo.buses, bounds=(0.95, 1.10), initialize=1.0)
modelo.theta = pyo.Var(modelo.scenarios, modelo.buses, bounds=(-0.5, 0.5), initialize=0.0)
modelo.Pg = pyo.Var(modelo.scenarios, [0, 1], initialize=0.4) # Gen 1 en bus 0, Gen 2 en bus 1
modelo.Qg = pyo.Var(modelo.scenarios, [0, 1], initialize=0.3)

# Fijar Slack (Nodo 3 -> índice 2 en Python)
for s in modelo.scenarios:
    modelo.theta[s, 2].fix(0.0)

# Límites de generación (pu)
p_lims = {0: (0, 3.0), 1: (0, 0.8)} 
for s in modelo.scenarios:
    for g_idx in [0, 1]:
        modelo.Pg[s, g_idx].setlb(p_lims[g_idx][0])
        modelo.Pg[s, g_idx].setub(p_lims[g_idx][1])

# --- Función Objetivo (Minimizar coste caso base) ---
# Coste G1 = 2 $/pu, Coste G2 = 1 $/pu
modelo.obj = pyo.Objective(expr= 2 * modelo.Pg['base', 0] + 1 * modelo.Pg['base', 1], sense=pyo.minimize)

# --- Restricciones de Flujo de Potencia (Balance AC) ---
def balance_P_rule(m, s, i):
    # Inyección de potencia
    p_gen = m.Pg[s, i] if i in [0, 1] else 0
    p_load = 0.8 if i == 2 else 0 # Carga en Nodo 3
    
    inyec = sum(m.V[s, i] * m.V[s, j] * (G_mat[i, j] * pyo.cos(m.theta[s, i] - m.theta[s, j]) + 
                                         B_mat[i, j] * pyo.sin(m.theta[s, i] - m.theta[s, j])) for j in m.buses)
    return p_gen - p_load == inyec

def balance_Q_rule(m, s, i):
    q_gen = m.Qg[s, i] if i in [0, 1] else 0
    q_load = 0.6 if i == 2 else 0
    
    inyec = sum(m.V[s, i] * m.V[s, j] * (G_mat[i, j] * pyo.sin(m.theta[s, i] - m.theta[s, j]) - 
                                         B_mat[i, j] * pyo.cos(m.theta[s, i] - m.theta[s, j])) for j in m.buses)
    return q_gen - q_load == inyec

modelo.c_balance_P = pyo.Constraint(modelo.scenarios, modelo.buses, rule=balance_P_rule)
modelo.c_balance_Q = pyo.Constraint(modelo.scenarios, modelo.buses, rule=balance_Q_rule)

# --- Restricciones de Capacidad de Línea (Flujo Aparente S) ---
def line_limit_rule(m, s, i, j):
    # Definir límites: S12_max = 0.25, S13=2, S23=2 (Base) / 0.4 (Cont)
    limit = 2.0
    if (i, j) == (0, 1) or (i, j) == (1, 0): limit = 0.25
    if s == 'cont' and ((i, j) == (1, 2) or (i, j) == (2, 1)): limit = 0.4 # Contingencia reduce capacidad en línea 2-3 a 0.4 pu
    
    # Flujo de i a j: Sij = Vi*conj(Iij) = Vi * conj(Yij*(Vi-Vj))
    # Aquí simplificamos con la magnitud compleja:
    p_ij = m.V[s, i]**2 * G_mat[i, j] - m.V[s, i]*m.V[s, j]*(G_mat[i, j]*pyo.cos(m.theta[s, i]-m.theta[s, j]) + B_mat[i, j]*pyo.sin(m.theta[s, i]-m.theta[s, j]))
    q_ij = -m.V[s, i]**2 * B_mat[i, j] - m.V[s, i]*m.V[s, j]*(G_mat[i, j]*pyo.sin(m.theta[s, i]-m.theta[s, j]) - B_mat[i, j]*pyo.cos(m.theta[s, i]-m.theta[s, j]))
    
    return p_ij**2 + q_ij**2 <= limit**2

modelo.c_line_limit = pyo.Constraint(modelo.scenarios, lines, rule=line_limit_rule)

# --- Restricciones de Acoplamiento (Rampas Correctivas) ---
# |Pg_cont - Pg_base| <= R
# --- Restricciones de Acoplamiento (Rampas Correctivas) ---

# G1: |Pg_cont - Pg_base| <= 3.0
def rampa_g1_up(m):
    return m.Pg['cont', 0] - m.Pg['base', 0] <= 3.0
modelo.c_ramp_g1_up = pyo.Constraint(rule=rampa_g1_up)

def rampa_g1_down(m):
    return m.Pg['cont', 0] - m.Pg['base', 0] >= -3.0
modelo.c_ramp_g1_down = pyo.Constraint(rule=rampa_g1_down)

# G2: |Pg_cont - Pg_base| <= 0.2
def rampa_g2_up(m):
    return m.Pg['cont', 1] - m.Pg['base', 1] <= 0.2
modelo.c_ramp_g2_up = pyo.Constraint(rule=rampa_g2_up)

def rampa_g2_down(m):
    return m.Pg['cont', 1] - m.Pg['base', 1] >= -0.2
modelo.c_ramp_g2_down = pyo.Constraint(rule=rampa_g2_down)


In [4]:

# --- FUNCIÓN PARA RESOLVER ---
def resolver_sistema(modo="SCOPF"):
    if modo == "OPF_NORMAL":
        # Desactivamos las restricciones de contingencia y rampas
        modelo.c_ramp_g1_up.deactivate()
        modelo.c_ramp_g1_down.deactivate()
        modelo.c_ramp_g2_up.deactivate()
        modelo.c_ramp_g2_down.deactivate()
        # Para el OPF normal, ignoramos el escenario de contingencia en los límites de línea
        
    else:
        # Activamos todo para el SCOPF. En este caso evaluamos el escenario de contingencia
        modelo.c_ramp_g1_up.activate()
        modelo.c_ramp_g1_down.activate()
        modelo.c_ramp_g2_up.activate()
        modelo.c_ramp_g2_down.activate()

    solver = pyo.SolverFactory('ipopt')
    solver.solve(modelo)
    
    # Retornamos un diccionario con los valores del CASO BASE
    return {
        "coste": pyo.value(modelo.obj),
        "p1": pyo.value(modelo.Pg['base', 0]),
        "p2": pyo.value(modelo.Pg['base', 1]),
        "q1": pyo.value(modelo.Qg['base', 0]),
        "q2": pyo.value(modelo.Qg['base', 1]),
        "v1": pyo.value(modelo.V['base', 0]),
        "v2": pyo.value(modelo.V['base', 1]),
        "v3": pyo.value(modelo.V['base', 2]),
    }

# 1. Ejecutamos el OPF Normal (Sin seguridad)
res_opf = resolver_sistema(modo="OPF_NORMAL")

# 2. Ejecutamos el SCOPF (Correctivo)
res_scopf = resolver_sistema(modo="SCOPF")

# --- IMPRESIÓN DE LA TABLA COMPARATIVA ---
import pandas as pd

data = {
    "Variable": ["Coste [$]", "p1* [pu]", "p2* [pu]", "q1* [pu]", "q2* [pu]", "v1* [pu]", "v2* [pu]", "v3* [pu]"],
    "OPF (sin seguridad)": [
        res_opf["coste"], res_opf["p1"], res_opf["p2"], 
        res_opf["q1"], res_opf["q2"], 
        res_opf["v1"], res_opf["v2"], res_opf["v3"]
    ],
    "SCOPF (correctivo)": [
        res_scopf["coste"], res_scopf["p1"], res_scopf["p2"], 
        res_scopf["q1"], res_scopf["q2"], 
        res_scopf["v1"], res_scopf["v2"], res_scopf["v3"]
    ]
}

df = pd.DataFrame(data)
pd.options.display.float_format = '{:.4f}'.format

print("\n" + "="*60)
print(f"{'COMPARATIVA DE RESULTADOS':^60}")
print("="*60)
print(df.to_string(index=False))
print("="*60)


                 COMPARATIVA DE RESULTADOS                  
 Variable  OPF (sin seguridad)  SCOPF (correctivo)
Coste [$]               0.8352              1.1876
 p1* [pu]               0.0300              0.3832
 p2* [pu]               0.7751              0.4212
 q1* [pu]               0.3131              0.3236
 q2* [pu]               0.3386              0.3202
 v1* [pu]               1.0972              1.1000
 v2* [pu]               1.1000              1.1000
 v3* [pu]               1.0661              1.0676
